In [ ]:
# Cell 1: Environment Setup

include("../scripts/dictionaries.jl")
include("../scripts/helpers.jl")
using .WorkflowHelpers
using OMJulia
using Plots, DataFrames, CSV

# --- Configuration ---

# 1. Root dynamic model to initialize (qualified package name)
MODEL = "MyIEEE14.IEEE14DisconnectLine"

# 2. Input and output directories
MODELS_DIR = abspath("models")
OUTPUT_DIR = abspath("outputs")
mkpath(OUTPUT_DIR)

# 3. Derived package/model names and paths
SOURCE_PACKAGE = split(MODEL, ".")[1]
MODEL_DIR = joinpath(MODELS_DIR, SOURCE_PACKAGE)
PATHS = package_workflow_paths(MODEL, MODEL_DIR, OUTPUT_DIR)

MODELS_PKG_PATH = PATHS.source_package_file

# Auxiliary package (input, from BuildAux) lives under models/
AUX_PACKAGE = PATHS.aux_package
AUX_ROOT_MODEL = PATHS.aux_root_model
AUX_DIR = joinpath(MODELS_DIR, AUX_PACKAGE)
AUX_PACKAGE_FILE = joinpath(AUX_DIR, "package.mo")

# Initialized package (output) is written under outputs/
INITIALIZED_PACKAGE = PATHS.initialized_package
INITIALIZED_DIR = PATHS.initialized_dir
INITIALIZED_ROOT_MODEL = PATHS.initialized_root_model
INITIALIZED_PACKAGE_FILE = PATHS.initialized_package_file
INITIALIZED_ORDER_FILE = PATHS.initialized_order_file

# 4. Output directory for OpenModelica build artifacts
RUN_OUTPUT_DIR = joinpath(OUTPUT_DIR, INITIALIZED_PACKAGE * "_outputs")
mkpath(RUN_OUTPUT_DIR)

# 5. Library paths
# Your Dynawo installation (used only for its Modelica Standard Library)
DYNAWO_DIR = "/home/clarafercas/dynawo"
MODELICA_PKG_PATH = "$DYNAWO_DIR/OpenModelica/lib/omlibrary/Modelica/package.mo"
# Dynawo Modelica library from this repo
DYNAWO_PKG_PATH = abspath("../dynawo_library/Dynawo/package.mo")

# 6. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = Dict{String, String}()

# 7. Variable to plot after the initialized simulation
PLOT_VARIABLE = "Load2.terminal.V.re"

In [ ]:
# Cell 2: OpenModelica Setup + Package Loading

# 1. Load the original dynamic package and validate the configuration against it
SourceOMC = OMJulia.OMCSession()
omc_call(SourceOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(SourceOMC, "loadModel(Complex)")
omc_call(SourceOMC, "loadModel(ModelicaServices)")
omc_call(SourceOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
omc_call(SourceOMC, "loadFile(\"$MODELS_PKG_PATH\")")
config = check_user_configuration_package(SourceOMC;
    model = MODEL,
    init_model_by_component = INIT_MODEL_BY_COMPONENT,
)
chain = config.model_chain

# 2. Load the auxiliary package
AuxOMC = OMJulia.OMCSession()
omc_call(AuxOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(AuxOMC, "loadModel(Complex)")
omc_call(AuxOMC, "loadModel(ModelicaServices)")
omc_call(AuxOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
omc_call(AuxOMC, "loadFile(\"$AUX_PACKAGE_FILE\")")
println("Checking the auxiliary root model...")
chk_aux = sendExpression(AuxOMC, "checkModel($AUX_ROOT_MODEL)", parsed=false)
println(chk_aux)

In [ ]:
# Cell 3: Package Name Mapping
INITIALIZED_NAME_MAP = initialized_name_map(chain, INITIALIZED_PACKAGE)

In [ ]:
# Cell 4: Simulate Auxiliary Package + Extract Initialization Values

# Build and simulate the auxiliary root model.
ModelicaSystem(AuxOMC, AUX_PACKAGE_FILE, AUX_ROOT_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(AuxOMC, resultfile = AUX_PACKAGE * "_res.mat")

# Extract initialization values per original class in the inheritance chain.
initializable_by_model = Dict{String, Dict{String, Dict{String, Any}}}()
init_values_by_model = Dict{String, Dict{String, Dict{String, Float64}}}()

for model in chain
    components = get_all_components(SourceOMC, model)
    initializable_components = get_initializable_components(components, INIT_MODEL_BY_COMPONENT)
    init_values_by_component = extract_all_initialization_values(AuxOMC, components, INIT_MODEL_BY_COMPONENT)

    initializable_by_model[model] = initializable_components
    init_values_by_model[model] = init_values_by_component
end

In [ ]:
# Cell 5: Build and save the initialized package
sendExpression(SourceOMC, "deleteClass($INITIALIZED_PACKAGE)")
omc_call(SourceOMC, "loadString(\"within ; package $INITIALIZED_PACKAGE end $INITIALIZED_PACKAGE;\")", parsed = false)

for model in chain
    initialized_model = INITIALIZED_NAME_MAP[model]
    initialized_name = split(initialized_model, ".")[end]

    println("Copying and initializing ", model, " -> ", initialized_model)
    omc_call(SourceOMC, "copyClass($model, \"$initialized_name\", $INITIALIZED_PACKAGE)")

    apply_initialization_modifiers!(
        SourceOMC,
        initialized_model,
        initializable_by_model[model],
        init_values_by_model[model],
        INIT_MODEL_BY_COMPONENT,
    )
end

isdir(INITIALIZED_DIR) && rm(INITIALIZED_DIR; recursive = true, force = true)
mkpath(INITIALIZED_DIR)

write_package_files!(
    INITIALIZED_PACKAGE_FILE,
    INITIALIZED_ORDER_FILE,
    INITIALIZED_PACKAGE,
    package_class_names(chain, INITIALIZED_NAME_MAP),
)

save_initialized_package_classes!(
    SourceOMC,
    chain,
    INITIALIZED_NAME_MAP,
    INITIALIZED_DIR,
)

# Load the generated initialized package into a fresh session and validate it
InitializedOMC = OMJulia.OMCSession()
omc_call(InitializedOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
omc_call(InitializedOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
omc_call(InitializedOMC, "loadFile(\"$INITIALIZED_PACKAGE_FILE\")")
println("Checking the initialized root model...")
chk_initialized = sendExpression(InitializedOMC, "checkModel($INITIALIZED_ROOT_MODEL)", parsed=false)
println(chk_initialized)

In [ ]:
# Cell 6: Final simulation of the initialized package

ModelicaSystem(
    InitializedOMC,
    INITIALIZED_PACKAGE_FILE,
    INITIALIZED_ROOT_MODEL,
    [MODELICA_PKG_PATH, DYNAWO_PKG_PATH],
    customBuildDirectory = RUN_OUTPUT_DIR,
)

simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_ROOT_MODEL)
initialized_resultfile_prefix = INITIALIZED_PACKAGE

sim_result = sendExpression(
    InitializedOMC,
    "simulate($INITIALIZED_ROOT_MODEL, outputFormat=\"csv\", fileNamePrefix=\"$initialized_resultfile_prefix\", simflags=\"$simflags\")",
    parsed = false,
)

println(sim_result)

initialized_resultfile = joinpath(
    getWorkDirectory(InitializedOMC),
    initialized_resultfile_prefix * "_res.csv",
)

In [ ]:
# Cell 7: Plot the initialized package response

result_columns = names(DataFrame(CSV.File(initialized_resultfile; limit = 0)))
PLOT_VARIABLE in result_columns ||
    error("PLOT_VARIABLE \"$PLOT_VARIABLE\" is not a variable in the simulation result.")
initialized_df = DataFrame(CSV.File(initialized_resultfile; select = ["time", PLOT_VARIABLE]))

plotlyjs()
p = plot(initialized_df[!, "time"], initialized_df[!, PLOT_VARIABLE], label = [PLOT_VARIABLE])
plot!(p, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p, "Initialized package response")
xlabel!(p, "Time (s)")
ylabel!(p, PLOT_VARIABLE)
